# RAG with Qdrant — Step by Step

We move the Clause Shop RAG from `InMemoryVectorStore` to **Qdrant**, a vector database that saves vectors to disk.

|                        | `InMemoryVectorStore`                     | Qdrant                                        |
| ---------------------- | ----------------------------------------- | --------------------------------------------- |
| Where vectors live     | Python RAM                                | Qdrant on disk                                |
| After the program ends | Gone, so the PDF is re-embedded every run | Saved, so you embed once and query many times |
| Filtering              | Limited                                   | Filter by metadata (page, source, ...)        |

**Key terms**

- **Collection**: like a table (`shop_handbook`).
- **Point**: one stored chunk, made of an ID, a **vector** (embedding), and a **payload** (text + metadata).
- **Vector config**: the vector **size** (must match the embedding model) and the **distance** metric (Cosine).

**Plan:** Step 0 Setup → Step 1 Ingest → Step 2 Retrieve & Generate → Step 3 Upgrades (loop, citations, filtering).

**Prerequisites:** Ollama running, with `gemma3:1b` and `nomic-embed-text` pulled, and `shop_handbook.pdf` in the same folder as this notebook.


## Step 0 — Setup

### 0.1 Install the packages

`qdrant-client` talks to Qdrant. `langchain-qdrant` provides `QdrantVectorStore`, which replaces `InMemoryVectorStore`.


In [ ]:
# uv add --active langchain langchain-text-splitters langchain-community pypdf
# uv add --active langchain-ollama langchain-qdrant qdrant-client

### 0.2 Configuration

We use **local mode** (Qdrant runs inside Python and saves to `./qdrant_data`), so Docker isn't needed.
To use the Docker server instead, set `QDRANT_URL = "http://localhost:6333"`.


In [ ]:
DATA_PATH = "shop_handbook.pdf"

CHAT_PROVIDER = "ollama"
CHAT_MODEL = "gemma3:1b"
EMBED_PROVIDER = "ollama"
EMBED_MODEL = "nomic-embed-text:latest"
TOP_K = 3

COLLECTION_NAME = "shop_handbook"
QDRANT_PATH = "./qdrant_data"

SYSTEM_PROMPT = (
    "You are a helpful Clause Shop assistant. "
    "Answer only from the provided context. "
    "If the answer is not in the context, say you don't know."
)

### 0.3 Check the embedding dimension

The collection's vector **size** must equal the embedding model's output size. Measure it instead of guessing; it should be **768** for `nomic-embed-text`.


In [ ]:
from langchain.embeddings import init_embeddings


embedd = init_embeddings(
    model=EMBED_MODEL,
    provider=EMBED_PROVIDER
)

EMBEDDING_DIM = len(embedd.embed_query("Hi"))
print("Embedding dimension:", EMBEDDING_DIM)

### 0.4 Connect to Qdrant

Local mode allows **one client per folder at a time**. While this notebook holds the client, `ingest.py` and `chat.py` can't open `./qdrant_data`. The last cell closes it.


In [ ]:
from qdrant_client import QdrantClient


client = QdrantClient(path=QDRANT_PATH)

print("Existing collections:", [c.name for c in client.get_collections().collections])

## Step 1 — Ingest: Load → Chunk → Store

### 1.1 Load the PDF

`PyPDFLoader` returns one `Document` per page, and each has `metadata["page"]` (counted from **0**).


In [ ]:
from langchain_community.document_loaders import PyPDFLoader


pages = PyPDFLoader(DATA_PATH).load_and_split()

print(f"Loaded {len(pages)} pages")
print(pages[0].metadata)

### 1.2 Chunk

The chunking is the same as before. Each chunk keeps its page's metadata.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)
chunks = text_splitter.split_documents(pages)

print(f"Created {len(chunks)} chunks")
print(chunks[0].page_content[:300])

### 1.3 Create the collection

If the collection already exists, we **delete it and rebuild it**. Otherwise `add_documents` would append and duplicate every chunk on each re-run.


In [ ]:
from qdrant_client.models import Distance, VectorParams


if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print("Deleted old collection")

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

print("Created collection:", COLLECTION_NAME)

### 1.4 Store the chunks

This replaces `InMemoryVectorStore.from_documents(...)`. `add_documents` embeds each chunk and stores it as a **point**.


In [ ]:
from langchain_qdrant import QdrantVectorStore


vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embedd,
)
vector_store.add_documents(chunks)

### 1.5 Verify

The point count should equal the chunk count.


In [ ]:
print("Points stored:", client.count(COLLECTION_NAME).count)
print("Chunks created:", len(chunks))

### 1.6 Look inside a point

This shows what Qdrant actually stores. Notice the payload: your metadata is **nested under `metadata`**, which matters for filtering in Step 3.


In [ ]:
points, _ = client.scroll(COLLECTION_NAME, limit=1, with_payload=True, with_vectors=True)
p = points[0]
print("ID:", p.id)
print("Vector length:", len(p.vector))
print("Payload keys:", list(p.payload.keys()))
print("Metadata:", p.payload["metadata"])
print("Text:", p.payload["page_content"][:200])

## Step 2 — Retrieve & Generate

In `chat.py` you'd connect to the existing collection **without re-loading the PDF**:

```python
vector_store = QdrantVectorStore(client=client, collection_name=COLLECTION_NAME, embedding=embedd)
```

In this notebook, `vector_store` is already connected. The embeddings are still needed, because the **question** must be embedded with the **same model**.

### 2.1 Retrieve

For Cosine distance, a **higher score means more similar**.


In [ ]:
query = "What is the return policy?"   # change to a question your handbook answers

results = vector_store.similarity_search_with_score(query, k=TOP_K)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"Rank {rank} | Score {score:.4f} | Page {doc.metadata['page'] + 1}")
    print(doc.page_content[:200], "\n")

### 2.2 Generate

The generation code is unchanged from the original RAG.


In [ ]:
from langchain.chat_models import init_chat_model


llm = init_chat_model(
    model=CHAT_MODEL,
    model_provider=CHAT_PROVIDER,
    temperature=0.1
)

retrieved_docs = [doc for doc, _ in results]
context = "\n\n".join(d.page_content for d in retrieved_docs)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"},
]
print("AI:", llm.invoke(messages).text)

## Step 3 — Upgrades

### 3.1 Page citations

1. Label each chunk with its page **inside the context**, so the model knows where text came from.
2. Print the source pages **from metadata**. This doesn't depend on the LLM, so it's always accurate.

`+ 1` converts PyPDFLoader's 0-based page numbers into normal page numbers.


In [ ]:
def ask(query, k=TOP_K, qfilter=None, debug=False):
    results = vector_store.similarity_search_with_score(query, k=k, filter=qfilter)
    docs = [d for d, _ in results]

    if debug:
        for rank, (d, s) in enumerate(results, start=1):
            print(f"  [{rank}] score={s:.4f} page={d.metadata['page'] + 1}")

    if not docs:
        print("AI: I don't know.")
        return

    context = "\n\n".join(f"[Page {d.metadata['page'] + 1}]\n{d.page_content}" for d in docs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    print("AI:", llm.invoke(messages).text)
    print("Sources: pages", sorted({d.metadata["page"] + 1 for d in docs}))

ask("What is the return policy?", debug=True)

In [ ]:
def ask(query, k=TOP_K, qfilter=None, debug=False):
    results = vector_store.similarity_search_with_score(query, k=k, filter=qfilter)
    docs = [d for d, _ in results]

    if debug:
        for rank, (d, s) in enumerate(results, start=1):
            print(f"  [{rank}] score={s:.4f} page={d.metadata['page'] + 1}")

    if not docs:
        print("AI: I don't know.")
        return

    context = "\n\n".join(f"[Page {d.metadata['page'] + 1}]\n{d.page_content}" for d in docs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ]
    print("AI:", llm.invoke(messages).text)
    print("Sources: pages", sorted({d.metadata["page"] + 1 for d in docs}))

ask("What is the return policy?", debug=True)

### 3.2 A question that's not in the handbook

The model should say it doesn't know.


In [ ]:
ask("What is the capital of Pakistan?")

### 3.3 Metadata filtering

Search only part of the data, for example the first 5 pages. The filter runs **inside Qdrant**, together with the vector search.

- The key is `metadata.page` (nested), not `page`.
- `Range` is for numbers. For an exact match, use `match=models.MatchValue(value=...)`, e.g. on `metadata.source` when you have several PDFs.


In [ ]:
from qdrant_client import models


first_5_pages = models.Filter(
    must=[models.FieldCondition(key="metadata.page", range=models.Range(lt=5))]
)

ask("What is the return policy?", qfilter=first_5_pages, debug=True)

### 3.4 Chat loop

This is what `chat.py` does. Setup happens once, and the loop handles each question. Type `exit` to stop.

Each question is **independent**: there's no chat memory yet, so follow-up questions like "what about sale items?" lose the earlier context.


In [ ]:
while True:
    q = input("You: ").strip()
    if q.lower() in {"exit", "quit", ""}:
        break
    ask(q)
    print()

## Clean up

Close the client to release the local-mode lock, so `ingest.py` and `chat.py` can open `./qdrant_data`.


In [ ]:
client.close()